# Seed Correlation Summary

This notebook computes pairwise score correlations across 3 seeds for 4 dataset cases and outputs a summary table.

**Cases:**
- MinneApple | with traditional aug
- MinneApple | without traditional aug
- GWHD | with traditional aug
- GWHD | without traditional aug

**Output:** One row per case showing mean Pearson and Spearman correlations across the 3 seeds.


In [1]:
from pathlib import Path
import json
import numpy as np


def rankdata(arr: np.ndarray) -> np.ndarray:
    """Convert array values to ranks."""
    order = np.argsort(arr)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(arr), dtype=float)
    _, inv, counts = np.unique(arr, return_inverse=True, return_counts=True)
    for i, c in enumerate(counts):
        if c > 1:
            idx = np.where(inv == i)[0]
            ranks[idx] = ranks[idx].mean()
    return ranks + 1.0


def safe_pearson(x: np.ndarray, y: np.ndarray) -> float:
    """Compute Pearson correlation safely."""
    if len(x) < 2 or len(y) < 2:
        return float("nan")
    if np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return float("nan")
    return float(np.corrcoef(x, y)[0, 1])


def safe_spearman(x: np.ndarray, y: np.ndarray) -> float:
    """Compute Spearman correlation safely."""
    if len(x) < 2 or len(y) < 2:
        return float("nan")
    rx = rankdata(x)
    ry = rankdata(y)
    return safe_pearson(rx, ry)


def load_score_map(score_path: str | Path) -> dict[str, float]:
    """Load image scores from score_results.json."""
    p = Path(score_path)
    if not p.exists():
        raise FileNotFoundError(f"Score file not found: {p}")

    with open(p, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_difficulties = data.get("image_difficulties", [])
    if not isinstance(image_difficulties, list):
        raise ValueError(f"Invalid image_difficulties in: {p}")

    score_map: dict[str, float] = {}
    for item in image_difficulties:
        raw_path = item.get("image_path", "")
        score = item.get("difficulty_score", None)
        if not raw_path or score is None:
            continue
        key = Path(raw_path).name
        score_map[key] = float(score)

    return score_map


In [2]:
# Define case-to-seed paths
case_to_seed_paths = {
    "MinneApple | with traditional aug": [
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/with_trad_aug/minneapple/scoring/seed_123/score_results.json"),
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/with_trad_aug/minneapple/scoring/seed_456/score_results.json"),
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/with_trad_aug/minneapple/scoring/seed_789/score_results.json"),
    ],
    "MinneApple | without traditional aug": [
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/no_trad_aug/minneapple/scoring/seed_123/score_results.json"),
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/no_trad_aug/minneapple/scoring/seed_456/score_results.json"),
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/no_trad_aug/minneapple/scoring/seed_789/score_results.json"),
    ],
    "GWHD | with traditional aug": [
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/with_trad_aug/gwhd/scoring/seed_123/score_results.json"),
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/with_trad_aug/gwhd/scoring/seed_456/score_results.json"),
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/with_trad_aug/gwhd/scoring/seed_789/score_results.json"),
    ],
    "GWHD | without traditional aug": [
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/no_trad_aug/gwhd/scoring/seed_123/score_results.json"),
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/no_trad_aug/gwhd/scoring/seed_456/score_results.json"),
        Path("/home/khanh/Projects/DifficultyAgri/.cache_result/no_trad_aug/gwhd/scoring/seed_789/score_results.json"),
    ],
}

rows = []

for case_name, seed_paths in case_to_seed_paths.items():
    if len(seed_paths) != 3:
        rows.append({
            "case": case_name,
            "n_common_images": 0,
            "pearson(123,456)": np.nan,
            "pearson(123,789)": np.nan,
            "pearson(456,789)": np.nan,
            "mean_pairwise_pearson": np.nan,
            "spearman(123,456)": np.nan,
            "spearman(123,789)": np.nan,
            "spearman(456,789)": np.nan,
            "mean_pairwise_spearman": np.nan,
        })
        continue

    # Try to load all 3 seed score maps
    try:
        score_maps = []
        for seed_path in seed_paths:
            if not seed_path.exists():
                raise FileNotFoundError(f"Missing seed file: {seed_path}")
            score_map = load_score_map(seed_path)
            if len(score_map) == 0:
                raise ValueError(f"Empty score map from: {seed_path}")
            score_maps.append(score_map)
    except Exception as e:
        print(f"Warning: Could not load all 3 seeds for {case_name}: {e}")
        rows.append({
            "case": case_name,
            "n_common_images": 0,
            "pearson(123,456)": np.nan,
            "pearson(123,789)": np.nan,
            "pearson(456,789)": np.nan,
            "mean_pairwise_pearson": np.nan,
            "spearman(123,456)": np.nan,
            "spearman(123,789)": np.nan,
            "spearman(456,789)": np.nan,
            "mean_pairwise_spearman": np.nan,
        })
        continue

    # Find common images across all 3 seeds
    common_images = set(score_maps[0].keys())
    for score_map in score_maps[1:]:
        common_images &= set(score_map.keys())
    
    common_images = sorted(common_images)
    n_common = len(common_images)

    if n_common < 2:
        print(f"Warning: Too few common images for {case_name}: {n_common}")
        rows.append({
            "case": case_name,
            "n_common_images": n_common,
            "pearson(123,456)": np.nan,
            "pearson(123,789)": np.nan,
            "pearson(456,789)": np.nan,
            "mean_pairwise_pearson": np.nan,
            "spearman(123,456)": np.nan,
            "spearman(123,789)": np.nan,
            "spearman(456,789)": np.nan,
            "mean_pairwise_spearman": np.nan,
        })
        continue

    # Build score matrix: [n_images, 3]
    score_matrix = np.array([
        [score_maps[i][img] for i in range(3)]
        for img in common_images
    ], dtype=float)

    # Compute pairwise correlations
    pairs = [(0, 1), (0, 2), (1, 2)]
    pearson_vals = []
    spearman_vals = []
    
    for i, j in pairs:
        x = score_matrix[:, i]
        y = score_matrix[:, j]
        pearson_vals.append(safe_pearson(x, y))
        spearman_vals.append(safe_spearman(x, y))

    mean_pearson = float(np.nanmean(pearson_vals)) if len(pearson_vals) > 0 else np.nan
    mean_spearman = float(np.nanmean(spearman_vals)) if len(spearman_vals) > 0 else np.nan

    rows.append({
        "case": case_name,
        "n_common_images": n_common,
        "pearson(123,456)": pearson_vals[0],
        "pearson(123,789)": pearson_vals[1],
        "pearson(456,789)": pearson_vals[2],
        "mean_pairwise_pearson": mean_pearson,
        "spearman(123,456)": spearman_vals[0],
        "spearman(123,789)": spearman_vals[1],
        "spearman(456,789)": spearman_vals[2],
        "mean_pairwise_spearman": mean_spearman,
    })

print(f"Processed {len(rows)} cases.")


Processed 4 cases.


In [4]:
# Output summary table
try:
    import pandas as pd
    table = pd.DataFrame(rows)
    
    # Format numeric columns to 4 decimal places
    for col in [
        "pearson(123,456)",
        "pearson(123,789)",
        "pearson(456,789)",
        "mean_pairwise_pearson",
        "spearman(123,456)",
        "spearman(123,789)",
        "spearman(456,789)",
        "mean_pairwise_spearman",
    ]:
        table[col] = table[col].map(lambda x: f"{x:.4f}" if not np.isnan(x) else "nan")
    
    display(table)
except Exception:
    # Fallback to plain text table
    header = [
        "case",
        "n_common_images",
        "pearson(123,456)",
        "pearson(123,789)",
        "pearson(456,789)",
        "mean_pairwise_pearson",
        "spearman(123,456)",
        "spearman(123,789)",
        "spearman(456,789)",
        "mean_pairwise_spearman",
    ]
    print(" | ".join(header))
    print("-" * 180)
    for r in rows:
        print(
            f"{r['case']} | {r['n_common_images']} | "
            f"{r['pearson(123,456)']:.4f} | {r['pearson(123,789)']:.4f} | {r['pearson(456,789)']:.4f} | {r['mean_pairwise_pearson']:.4f} | "
            f"{r['spearman(123,456)']:.4f} | {r['spearman(123,789)']:.4f} | {r['spearman(456,789)']:.4f} | {r['mean_pairwise_spearman']:.4f}"
        )


,case,n_common_images,"pearson(123,456)","pearson(123,789)","pearson(456,789)",mean_pairwise_pearson,"spearman(123,456)","spearman(123,789)","spearman(456,789)",mean_pairwise_spearman
0,MinneApple | with traditional aug,536,0.8254,0.8088,0.7845,0.8062,0.8287,0.8195,0.8000,0.8161
1,MinneApple | without traditional aug,536,0.6738,0.6652,0.6794,0.6728,0.6146,0.5833,0.6401,0.6127
2,GWHD | with traditional aug,2700,0.9048,0.8927,0.9029,0.9001,0.8892,0.8922,0.8790,0.8868
3,GWHD | without traditional aug,2700,0.8648,0.8556,0.8697,0.8634,0.8888,0.9013,0.8833,0.8911
